In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import math
import re
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile

import sys
# sys.path.append('../..')
# from mount_drive import mount_s_drive

In [3]:
# mount_s_drive(subfolder='LCICM/Databases')

In [4]:
def nameSearchCardiacArrest(text):
    text = str(text).lower()
    patterns = [r'arrest',
                r'cardia.*rrest',
                r'cardio.*rrest',
                r'circulat.*rrest',
                r'\basystole',
                r'\basystolia',
                r'\bpea\b|pulseless elec.*act.*',
                r'post.*rrest']
    if any(re.search(pattern, text) for pattern in patterns):
        exclusion_patterns = [r'history|hx|h/o',
                              r'before cardiac arrest',
                              r'due to cardiac arrest',
                              r'neonatal',
                              r'newborn',
                              r'resp.*rrest']       
        if not any(re.search(pattern, text) for pattern in exclusion_patterns):
            return True   
    return False

def icdSearchCardiacArrest(text):
    text = str(text).lower()
    icd10_match = re.search(r'\bi46.*\b', text)
    icd9_match = re.search(r'\b4275\b', text)
    return bool(icd10_match or icd9_match)

In [5]:
def read_by_chunks(file_path,chunk_size=1e6,head='infer',compression=None):
    df_chunks = []
    num_chunks = 0
    
    for chunk in pd.read_csv(file_path,chunksize=chunk_size,header=head,compression=compression):
        df_chunks.append(chunk)
        if num_chunks % 20 == 0:
            print(f'Chunk {num_chunks} Processed.')
        num_chunks += 1
        del chunk
    print('Processing finished.')
    
    return pd.concat(df_chunks, ignore_index=True)

In [6]:
database_folder = '/projects/LCICM/'

In [7]:
dx_df = pd.read_csv(database_folder+'mimic-iv-2.2/hosp/d_icd_diagnoses.csv')
diagnoses_df = read_by_chunks(database_folder+'mimic-iv-2.2/hosp/diagnoses_icd.csv')
ed_diagnoses_df = read_by_chunks(database_folder+'mimic-iv-ed-2.2/mimic-iv-ed-2.2/ed/diagnosis.csv.gz', compression='gzip')
procedureevents_df = read_by_chunks(database_folder+'mimic-iv-2.2/icu/procedureevents.csv')
patients_df = pd.read_csv(database_folder+'mimic-iv-2.2/hosp/patients.csv')
admissions_df = pd.read_csv(database_folder+'mimic-iv-2.2/hosp/admissions.csv')
icustays_df = pd.read_csv(database_folder+'mimic-iv-2.2/icu/icustays.csv')
edstays_df = read_by_chunks(database_folder+'mimic-iv-ed-2.2/mimic-iv-ed-2.2/ed/edstays.csv.gz', compression='gzip')

Chunk 0 Processed.
Processing finished.
Chunk 0 Processed.
Processing finished.
Chunk 0 Processed.
Processing finished.
Chunk 0 Processed.
Processing finished.


In [8]:
dx_df['icd_code'] = dx_df['icd_code'].str.strip()
dx_df['long_title'] = dx_df['long_title'].str.strip()
dx_df['icd_search'] = dx_df['icd_code'].transform(icdSearchCardiacArrest)
dx_df['name_search'] = dx_df['long_title'].transform(nameSearchCardiacArrest)
ca_dx_df = dx_df[(dx_df['icd_search'])|(dx_df['name_search'])]

In [9]:
diagnoses_df['icd_code'] = diagnoses_df['icd_code'].str.strip()
ca_df = diagnoses_df.merge(ca_dx_df[['icd_code','long_title']], on='icd_code', how='inner')
ca_df = ca_df[~ca_df['long_title'].str.lower().str.contains('intraoperative')]
ca_patients_df = patients_df[['subject_id','gender','anchor_age']].merge(ca_df, on='subject_id', how='right')
ca_patients_df = ca_patients_df[ca_patients_df['anchor_age']>=18]

In [10]:
ed_diagnoses_df['icd_code'] = ed_diagnoses_df['icd_code'].str.strip()
ed_ca_df = ed_diagnoses_df.merge(ca_dx_df[['icd_code','long_title']], on='icd_code', how='inner')
ed_ca_df = ed_ca_df[~ed_ca_df['long_title'].str.lower().str.contains('intraoperative')]
ed_ca_patients_df = patients_df[['subject_id','gender','anchor_age']].merge(ed_ca_df, on='subject_id', how='right')
ed_ca_patients_df = ed_ca_patients_df[ed_ca_patients_df['anchor_age']>=18]
ed_ca_patients_df = ed_ca_patients_df.drop(columns=['icd_title'])
ed_ca_patients_df = ed_ca_patients_df.merge(edstays_df[['subject_id','hadm_id','stay_id']], on=['subject_id','stay_id'], how='left')
ed_ca_patients_df = ed_ca_patients_df.dropna(subset=['hadm_id'])
ed_ca_patients_df['stay_id'] = ed_ca_patients_df['hadm_id']
ed_ca_patients_df = ed_ca_patients_df.drop(columns=['hadm_id'])
ed_ca_patients_df = ed_ca_patients_df.rename(columns={'stay_id':'hadm_id'})

In [11]:
merged_df = pd.concat([ca_patients_df,ed_ca_patients_df])

In [12]:
ca_time_df = procedureevents_df[procedureevents_df['itemid']==225466]
ca_time_df = ca_time_df.sort_values(['subject_id','hadm_id','starttime'])
ca_time_df = ca_time_df.drop_duplicates('hadm_id', keep='first').reset_index(drop=True)
duplicate_subjects = ca_time_df.groupby('subject_id')['hadm_id'].nunique()
duplicate_subjects = duplicate_subjects[duplicate_subjects > 1].index
ca_time_df = ca_time_df[~ca_time_df['subject_id'].isin(duplicate_subjects)]

In [13]:
ca_time_df2 = patients_df[['subject_id','gender','anchor_age']].merge(ca_time_df[['subject_id','hadm_id','stay_id','starttime']],on='subject_id',how='right')
ca_time_df2 = ca_time_df2.merge(merged_df[['hadm_id','long_title']],on='hadm_id',how='left')
ca_time_df2 = ca_time_df2.merge(admissions_df[['subject_id','hadm_id','hospital_expire_flag']],on=['subject_id','hadm_id'],how='left')
ca_time_df2 = ca_time_df2.merge(icustays_df[['subject_id','hadm_id','stay_id','intime','outtime']],on=['subject_id','hadm_id','stay_id'],how='left')
ca_time_df2 = ca_time_df2.sort_values(['subject_id','hadm_id','intime'])
ca_time_df2 = ca_time_df2.drop_duplicates('subject_id', keep='first').reset_index(drop=True)
ca_time_df2['starttime'] = pd.to_datetime(ca_time_df2['starttime'], errors='coerce')
ca_time_df2['intime'] = pd.to_datetime(ca_time_df2['intime'], errors='coerce')
ca_time_df2['max_time'] = ca_time_df2[['starttime', 'intime']].max(axis=1)
#ca_time_df2 = ca_time_df2.drop(columns=['starttime','intime'])

In [14]:
ca_time_df2.head()

,subject_id,gender,anchor_age,hadm_id,stay_id,starttime,long_title,hospital_expire_flag,intime,outtime,max_time
0,10056037,F,68,21758160,30687961,2131-03-31 03:20:00,NaN,1,2131-03-30 21:51:40,2131-03-31 08:48:41,2131-03-31 03:20:00
1,10076958,F,64,28621351,39735886,2113-05-03 09:10:00,NaN,1,2113-05-03 04:16:55,2113-05-03 14:54:15,2113-05-03 09:10:00
2,10109956,F,62,26022059,37631738,2184-05-14 04:10:00,"Cardiac arrest, cause unspecified",0,2184-05-09 14:57:38,2184-05-23 22:23:29,2184-05-14 04:10:00
3,10137856,M,72,24697159,35659646,2186-05-17 22:17:00,Cardiac arrest,1,2186-05-17 09:02:28,2186-05-18 07:12:17,2186-05-17 22:17:00
4,10142197,F,75,20108756,36313195,2161-01-04 00:04:00,NaN,1,2161-01-03 16:01:00,2161-01-04 07:55:23,2161-01-04 00:04:00


In [15]:
ca_time_df2.to_csv('CA_time.csv',index=False)